In [0]:
%run /Workspace/Users/lupsecristian2000@gmail.com/Robotics/pipeline-finetune-gr00t-ssh/secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_2", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_2", 0o600)

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
echo == OS ==
. /etc/os-release
echo $PRETTY_NAME
echo == GPU / Driver ==
nvidia-smi
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
python3 --version
pip3 install h5py opencv-python-headless ffmpeg-python static-ffmpeg imageio pyarrow
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
sudo apt update
sudo apt install ffmpeg -y
ffmpeg -version
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
mkdir hdf5_datasets lerobot_datasets
EOF

In [0]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf5_datasets/$HDF5_DATASET_NAME.hdf5 \
  ubuntu@$BREV_IP:~/hdf5_datasets/

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
mkdir -p ~/lerobot_datasets/meta
mkdir -p ~/lerobot_datasets/data
mkdir -p ~/lerobot_datasets/videos/observation.images.top
mkdir -p ~/lerobot_datasets/videos/observation.images.wrist
ls -laR ~/lerobot_datasets/
EOF

In [0]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf52lerobot_script_files/convert_videos.py \
  ubuntu@$BREV_IP:~/convert_videos.py

In [0]:
%sh
scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
  /Volumes/workspace/default/hdf52lerobot_script_files/data_and_meta_convert.py \
  ubuntu@$BREV_IP:~/data_and_meta_convert.py

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP \
  "sed -i 's/TASK_NAME = \"your_task_name\"/TASK_NAME = \"$TASK_NAME\"/' ~/data_and_meta_convert.py"

In [0]:
%sh
ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << 'EOF'
tmux kill-session -t work 2>/dev/null
tmux new-session -d -s work
tmux send-keys -t work "python3 ~/convert_videos.py 2>&1 | tee ~/convert_videos.log && python3 ~/data_and_meta_convert.py 2>&1 | tee ~/data_and_meta_convert.log" Enter
tmux send-keys -t work "exit" Enter
EOF

In [0]:
%sh
echo "Waiting for data conversion to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t work 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Data conversion complete!"
    break
  fi
  
  sleep 60
done

ssh -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no ubuntu@$BREV_IP << EOF
cd \$HOME/lerobot_datasets/
du -sh .
ls -lh
EOF

In [0]:
# %sh
# scp -i /tmp/ssh_private_key_2 -o StrictHostKeyChecking=no \
#   ubuntu@$BREV_IP:~/lerobot_datasets/data/chunk-000/episode_000009.parquet \
#   /Volumes/workspace/default/datasets/ep9.parquet

In [0]:
import requests
import os
NEXT_JOB_ID = 449953930494301  

response = requests.post(
    f"{os.environ['DATABRICKS_HOST']}/api/2.1/jobs/run-now",
    headers={"Authorization": f"Bearer {os.environ['DATABRICKS_TOKEN']}"},
    json={"job_id": NEXT_JOB_ID}
)

if response.status_code == 200:
    print(f"Job 2 triggered: run_id={response.json()['run_id']}")
else:
    raise Exception(f"Failed to trigger Job 2: {response.text}")